# MADRL Training Notebook

This notebook configures, trains, evaluates, and visualizes multi-agent energy storage experiments. Run the cells from top to bottom.


In [ ]:
from pathlib import Path
import sys

# 启用 autoreload, 修改源码后自动重新加载模块, 方便开发调试
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    pass

# 从当前工作目录向上逐级搜索, 直到找到包含 configs 目录的项目根目录
project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "configs").exists():
    project_root = project_root.parent
if not (project_root / "configs").exists():
    raise RuntimeError("Could not locate the project root.")
# 将项目根目录加入 sys.path, 确保后续 import 能找到项目模块
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
project_root

In [ ]:
# 导入实验工具函数: 构建runner、评估、IO检查、配置摘要等
from scripts.utils.experiment_notebook_utils import (
    build_runner,
    evaluate_runner,
    get_lstm_artifact_root,
    get_madrl_checkpoint_root,
    inspect_runner_io,
    summarize_cfg,
)
# 导入 PyTorch 运行时配置工具 (设备选择、随机种子等)
from scripts.utils.torch_runtime import configure_torch_runtime, describe_device
# 导入实验配置组合函数
from configs import compose_experiment_config
# 导入可视化绘图工具: 价格-动作-SOC轨迹图 和 奖励分解图
from scripts.plots.plots import plot_last_k_episodes_price_action_soc
from scripts.plots.reward_plots import plot_reward_decomposition

In [ ]:
# Notebook layer: keep only experiment-specific knobs here.
# Use profiles for training-scale templates; keep only run-specific toggles in the notebook.
profile = "debug"                    # 训练规模模板: base/debug/fast_train
algorithm = "MADDPG"                 # 强化学习算法: MADDPG | MATD3
model_family = "mlp"                 # 策略/评论家网络骨架: mlp | transformer | graph
reward_type = "composite"            # 奖励函数设计: composite(组合奖励) | sparse(稀疏奖励)
future_horizon = 24                   # 规划时间窗口(15分钟为一步), 24步 = 6小时
sequence_source = "truth"            # 序列特征来源: truth(真实数据) | lstm(LSTM预测)
forecast_type = "perfect" if sequence_source == "truth" else "lstm"  # 根据序列来源确定预测类型
observation_profile = "simbench"     # 观测预设, 决定局部/序列特征的默认组合
local_features = None                 # None 表示使用 observation_profile 的默认局部特征
sequence_features = None              # None 表示使用 observation_profile 的默认序列特征
vec_env_type = "dummy"               # 向量化环境类型: dummy(单进程) | subproc(多进程)
runtime_mode = "performance"         # 运行模式: performance(高性能) | strict_reproducibility(严格可复现)
device_request = None                 # None 表示自动选择 CUDA, 也可手动指定设备
require_cuda = False                  # True 则在无 CUDA 时直接报错
seed = 0                              # 随机种子, 控制实验可复现性
n_eval_episodes = 2                   # 训练结束后的评估轮数
reward_plot_window = 20               # 奖励曲线的滑动平均窗口大小
n_recent_episodes_to_plot = 2         # 绘制最近几个 episode 的轨迹图

In [ ]:
# 根据上方超参数组合生成完整的实验配置对象
cfg = compose_experiment_config(
    profile=profile,
    algorithm=algorithm,
    model_family=model_family,
    reward_type=reward_type,
    observation_profile=observation_profile,
    forecast_type=forecast_type,
    vec_env_type=vec_env_type,
    local_features=local_features,
    sequence_features=sequence_features,
    data_dir=project_root / "data",
    runtime_mode=runtime_mode,
    seed=seed,
    require_cuda=require_cuda,
)

# 配置 PyTorch 运行时: 设备分配、随机种子、确定性设置等
runtime_state = configure_torch_runtime(
    cfg,
    device=device_request,
    seed=seed,
    require_cuda=require_cuda,
)
# 获取设备描述信息(GPU型号、显存等)
device_info = describe_device(runtime_state)

# Notebook overrides: keep only task-specific settings here.
# Put reusable training-scale settings such as batch_size, buffer_size, and num_envs back into profiles.
cfg.data.dataset_type = "csv_prosumer"               # 数据集类型: 使用 CSV 格式的产消者数据
cfg.train.num_envs = 12                               # 并行环境数量
cfg.train.train_episodes = 360                         # 总训练 episode 数
cfg.env.num_agents = 3                                 # 智能体数量, 需与数据集列数匹配
cfg.env.episode_limit = 96 * 2                         # 每个 episode 的环境步数, 96步/天 * 2天
cfg.env.future_horizon = future_horizon                # 序列观测和奖励前瞻的时间窗口
cfg.obs.local_features = ["time", "price", "load", "pv", "soc"]  # 局部观测特征列表
cfg.obs.sequence_features = ["price", "load", "pv"]   # 序列观测特征列表
cfg.forecast.target_signals = ["price", "load", "pv"] # 预测目标信号列表
# 若使用 LSTM 预测模式, 需要更长的历史窗口并指定预训练模型路径
if forecast_type == "lstm":
    cfg.forecast.history_window = max(cfg.forecast.history_window, 96 * 7)  # 至少7天历史数据
    cfg.forecast.lstm_artifact_root = get_lstm_artifact_root(project_root)

# 生成配置摘要, 方便检查实验设置是否正确
summary = summarize_cfg(cfg)
summary["device_info"] = device_info
summary

In [ ]:
# 构建训练 runner: 内部创建向量化环境、MADRL控制器、经验回放缓冲区等
runner = build_runner(cfg, seed=seed, env_name="NotebookTrain", number=1)
# 获取评估环境的奖励函数, 用于后续奖励分解绘图
plot_reward_fn = runner.env_evaluate.reward_fn
# 打印观测空间结构信息, 确认特征维度正确
print("Observation schema =", cfg.runtime.observation_schema)
print("Observation layout =", cfg.runtime.observation_layout)
print("Action dim =", cfg.runtime.action_dim)

In [ ]:
# IO 健全性检查: 执行一次环境 step, 验证观测/动作/奖励的形状是否符合预期
sanity_summary = inspect_runner_io(runner, cfg)
sanity_summary

In [ ]:
# 启动训练循环: 在向量化环境中按 episode 迭代, 收集经验并更新策略网络
episodes_completed = runner.run()
print("训练完成 episode 数 =", episodes_completed)
# 输出性能摘要: 包含总耗时、每步耗时、更新耗时等关键指标
runner.perf_summary

In [ ]:
# 使用确定性策略对训练好的模型进行评估
eval_results = evaluate_runner(runner, cfg, n_episodes=n_eval_episodes, deterministic=True)
print("Mean evaluation reward =", eval_results["mean_episode_reward"])

# 获取评估历史记录, 用于绘图
eval_histories = eval_results.get("histories", [])
# 评估窗口不超过实际评估 episode 数
eval_window = min(reward_plot_window, max(1, len(eval_results["episode_rewards"])))

# 绘制训练阶段的奖励分解图: 展示各奖励分量随训练的变化趋势
plot_reward_decomposition(
    history=runner.history,
    episode_rewards=runner.episode_rewards,
    reward_fn=plot_reward_fn,
    title="Training Reward Decomposition",
    window=reward_plot_window,
)

# 绘制训练阶段最近几个 episode 的价格-动作-SOC轨迹图
plot_last_k_episodes_price_action_soc(
    history=runner.history,
    k=n_recent_episodes_to_plot,
    n_agents=cfg.env.num_agents,
    title_prefix="Train",
)

# 绘制评估阶段的奖励分解图
plot_reward_decomposition(
    history=eval_histories,
    episode_rewards=eval_results["episode_rewards"],
    reward_fn=plot_reward_fn,
    title="Evaluation Reward Decomposition",
    window=eval_window,
)

# 绘制评估阶段的价格-动作-SOC轨迹图
plot_last_k_episodes_price_action_soc(
    history=eval_histories,
    k=min(n_recent_episodes_to_plot, len(eval_histories)),
    n_agents=cfg.env.num_agents,
    title_prefix="Eval",
)

In [ ]:
# 获取模型检查点保存路径并创建目录
save_dir = get_madrl_checkpoint_root(project_root)
save_dir.mkdir(parents=True, exist_ok=True)
# 保存训练好的模型参数(actor和critic网络权重)
runner.save_model(str(save_dir), episode=episodes_completed)

# 汇总最终实验结果: 保存路径、episode数、评估奖励、性能指标
final_summary = {
    "save_dir": str(save_dir),
    "saved_episode": int(episodes_completed),
    "mean_eval_reward": float(eval_results["mean_episode_reward"]),
    "perf_summary": runner.perf_summary,
}
print("模型已保存到 =", save_dir)
# 关闭 runner, 释放环境和 GPU 资源
runner.close()
final_summary